# 03 -- Cross-PTA consistency checks (stretch goal)

**Optional, ~10 min if there is time.**

Once we have a consistent MetaPulsar (notebook 02), the natural next question is: do the per-PTA pieces actually agree on the noise model and on the timing-model parameters they share? This is exactly the question that `metapulsar.consistency` answers, using `tensiometer`-style parameter-shift estimators on top of single-PTA posteriors.

**Realistic production workflow.**

```python
from metapulsar.consistency import (
    list_ptas, subset_metapulsar, build_fftint_posterior,
    samples_to_mcsamples, hyper_tension, waveform_tension, timing_tension, summarize,
)

# 1. Slice the MetaPulsar back into per-PTA enterprise pulsars (no rebuild).
per_pta_psr = {pta: subset_metapulsar(mp, pta) for pta in list_ptas(mp)}
# 2. For each PTA, build an FFTInt RN+DM posterior (discovery + NumPyro NUTS).
#    *This is the slow step* -- minutes per PTA per pulsar.
posteriors = {
    pta: build_fftint_posterior(psr, model="rn+dm", n_modes=30, backend="discovery")
    for pta, psr in per_pta_psr.items()
}
```

Running NUTS live in a tutorial is not realistic. We replace step (2) with **synthetic, representative posteriors** drawn around plausible injected values. Every other API call -- `samples_to_mcsamples`, `hyper_tension`, `waveform_tension`, `timing_tension`, `summarize` -- is exactly what a production pipeline runs.

**Optional dependencies.** `metapulsar.consistency` lazy-imports `getdist` and `tensiometer` only when you actually call into them. If `tensiometer` is missing the non-Gaussian estimator gracefully degrades to the closed-form Gaussian shift; `getdist` is required for `samples_to_mcsamples` (typically pulled in transitively via `tensiometer`).

## Step 1 -- Rebuild the consistent MetaPulsar from notebook 02's file-data dict

MetaPulsar objects hold open file handles / thread locks and are not picklable, so notebook 02 only persisted the plain `FILE_DATA_REGISTRY` dict. We rebuild the target pulsar here -- the same `create_metapulsar(..., combination_strategy="consistent")` call as in notebook 02, ~30-60 s. If `FILE_DATA_REGISTRY` is not in the IPython store, run notebook 02 first.

In [1]:
import sys
import warnings
from pathlib import Path

import loguru
import numpy as np

from metapulsar import create_metapulsar

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)


def quiet_loguru(level: str = "WARNING") -> None:
    loguru.logger.remove()
    loguru.logger.add(sys.stdout, level=level)


quiet_loguru()

%store -r FILE_DATA_REGISTRY
%store -r TUTORIAL_TARGET

TARGET = TUTORIAL_TARGET
file_data = FILE_DATA_REGISTRY[TARGET]

mp = create_metapulsar(
    file_data=file_data,
    combination_strategy="consistent",
    parfile_output_dir="./parfiles",
)
quiet_loguru()

print(f"Rebuilt consistent MetaPulsar for {TARGET}")
print(f"  PTAs : {list(mp._pulsars.keys())}")
print(f"  TOAs : {len(mp.toas)}")

/opt/venvs/pta/lib/python3.12/site-packages/enterprise/signals/utils.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import Requirement, resource_filename


2026-04-22 08:33:14.083 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:33:14.086 | WARNING  | metapulsar.position_helpers:_extract_ecliptic_coordinates_optimized:540 - Missing PMELONG/PMELAT or POSEPOCH in parfile. Using catalogued position without proper motion propagation. Canonical naming may be unstable across epochs.


2026-04-22 08:33:14.266 | WARNING  | pint.models.model_builder:choose_binary_model:622 - Found T2 binary model. Gracefully converting T2 to: BT.


2026-04-22 08:33:14.272 | WARNING  | pint.models.model_builder:__call__:224 - UNITS is not specified. Assuming TDB...


...

Rebuilt consistent MetaPulsar for J1853+1303
  PTAs : ['EPTA dr2', 'NANOGrav 9y']
  TOAs : 1470


---------------------------------------------------------------------------------------------------


[textOutput.C:308] Notice: Parameter uncertainties NOT multiplied by sqrt(red. chisq)


Jump 1 (                -sys JBO.DFB.1520 0 1): 0 0 Y


Jump 2 (                -sys NRT.BON.1600 0 1): 0 0 Y


## Step 2 -- Slice the MetaPulsar back into per-PTA pulsars

`subset_metapulsar` is a pure dictionary lookup on the per-PTA enterprise pulsars that the MetaPulsar already retains internally -- no model rebuild, no PINT re-run. This is what makes the consistency-check pipeline cheap to set up: the slow part is the per-PTA NUTS run, not the slicing.

In [2]:
from metapulsar import consistency

ptas = consistency.list_ptas(mp)
per_pta_psr = {pta: consistency.subset_metapulsar(mp, pta) for pta in ptas}

for pta, psr in per_pta_psr.items():
    print(f"  {pta:<15}  toas={len(psr.toas):>6d}  freq_min={psr.freqs.min():.0f} MHz")

  NANOGrav 9y      toas=  1369  freq_min=422 MHz
  EPTA dr2         toas=   101  freq_min=1398 MHz


Binary model: T2


Mass function                  = 0.005439633929 


Minimum, median and maximum companion mass: 0.2395 < 0.2814 < 0.6379 solar masses


Total proper motion = 3.3094 +/- 0 mas/yr


Total time span = 3066.455 days = 8.395 years


## Step 3 -- Synthetic per-PTA posteriors

We draw three sets of posterior samples per PTA: the RN power-law hyperparameters, the DM power-law hyperparameters, and a small set of merged timing-model parameters that all three PTAs constrain (here: position + spin). Every PTA gets samples drawn around the same injected truth with somewhat different per-PTA widths -- this is the *null* hypothesis (the PTAs agree).

In [3]:
INJECTED_RN = {"log10_A": -13.5, "gamma": 3.0}
INJECTED_DM = {"log10_A": -13.5, "gamma": 2.5}
INJECTED_TM = {
    "d_RAJ": 0.0,
    "d_DECJ": 0.0,
    "d_F0": 0.0,
    "d_F1": 0.0,
}


def gaussian_samples(mean, sigma, n_samples=4000, rng=None):
    rng = rng or np.random.default_rng(0)
    return rng.normal(mean, sigma, n_samples)


def synth_hyper(mean, sigma_logA=0.15, sigma_gamma=0.4, n_samples=4000, rng=None):
    return {
        "log10_A": gaussian_samples(mean["log10_A"], sigma_logA, n_samples, rng),
        "gamma": gaussian_samples(mean["gamma"], sigma_gamma, n_samples, rng),
    }


def synth_waveform(n_modes=15, n_samples=4000, rng=None):
    rng = rng or np.random.default_rng(0)
    return {
        f"{kind}_{k:02d}": rng.normal(0.0, 1.0, n_samples)
        for k in range(n_modes)
        for kind in ("c", "s")
    }


def synth_timing(mean, sigmas, n_samples=4000, rng=None):
    return {name: gaussian_samples(mean[name], sigmas[name], n_samples, rng) for name in mean}


TM_SIGMAS = {"d_RAJ": 1.0e-9, "d_DECJ": 2.0e-9, "d_F0": 5.0e-13, "d_F1": 1.0e-20}

rn_chains = {}
dm_chains = {}
wf_chains = {}
tm_chains = {}
for ii, pta in enumerate(ptas):
    rng = np.random.default_rng(100 + ii)
    rn_chains[pta] = consistency.samples_to_mcsamples(
        synth_hyper(INJECTED_RN, rng=rng), label=f"{pta} RN"
    )
    dm_chains[pta] = consistency.samples_to_mcsamples(
        synth_hyper(INJECTED_DM, sigma_logA=0.18, sigma_gamma=0.5, rng=rng),
        label=f"{pta} DM",
    )
    wf_chains[pta] = consistency.samples_to_mcsamples(
        synth_waveform(n_modes=15, rng=rng), label=f"{pta} waveform"
    )
    tm_chains[pta] = consistency.samples_to_mcsamples(
        synth_timing(INJECTED_TM, TM_SIGMAS, rng=rng), label=f"{pta} timing"
    )

print("Built synthetic getdist.MCSamples chains for:", list(rn_chains.keys()))
print("Per-chain example: log10_A samples for", ptas[0], "->", rn_chains[ptas[0]].samples[:3, 0])

Electron density (1AU) 4


Solar system ephem     DE421


Time scale             TT(BIPM2011)


Binary model           T2


In here writing a new parameter file: /tmp/tmp7oc_kiyf.par


Notice: There were 1 warnings. Summaries are shown below, check logs for full details.


Warning #1: [TIM1] Please place MODE flags in the parameter file 


Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Built synthetic getdist.MCSamples chains for: ['NANOGrav 9y', 'EPTA dr2']
Per-chain example: log10_A samples for NANOGrav 9y -> [-13.67363245 -13.45653663 -13.38287189]


## Step 4 -- Pairwise tensions

We loop over PTA pairs and compute three tensions per pair:

* `hyper_tension(...)` on `(log10_A, gamma)` for RN and DM separately;
* `waveform_tension(...)` on the 30 Fourier coefficients (c_k, s_k) for k=0..14;
* `timing_tension(...)` on the merged timing-model parameters.

All three return a `TensionResult(n_sigma, p_value, method, n_params, extra)` dataclass. With synthetic samples drawn around the same injected truth, every n_sigma should be small (well below 1).

In [4]:
coef_names = [f"{kind}_{k:02d}" for k in range(15) for kind in ("c", "s")]
tm_params = list(INJECTED_TM)

rows = []
for i, pta_a in enumerate(ptas):
    for pta_b in ptas[i + 1 :]:
        rn = consistency.hyper_tension(rn_chains[pta_a], rn_chains[pta_b])
        dm = consistency.hyper_tension(dm_chains[pta_a], dm_chains[pta_b])
        wf = consistency.waveform_tension(
            wf_chains[pta_a], wf_chains[pta_b], coef_names=coef_names, method="gaussian"
        )
        tm = consistency.timing_tension(
            tm_chains[pta_a], tm_chains[pta_b], params=tm_params, method="gaussian"
        )
        for check, result in (
            ("hyper_RN", rn),
            ("hyper_DM", dm),
            ("waveform_RN", wf),
            ("timing", tm),
        ):
            rows.append(
                dict(pulsar=TARGET, pta_a=pta_a, pta_b=pta_b, check=check, **result.as_dict())
            )

for row in rows:
    print(
        f"{row['pulsar']}  {row['pta_a']:>15} vs {row['pta_b']:<15}  {row['check']:<12}  "
        f"n_sigma={row['n_sigma']:>7.3f}  method={row['method']}"
    )

J1853+1303      NANOGrav 9y vs EPTA dr2         hyper_RN      n_sigma=  0.030  method=kde
J1853+1303      NANOGrav 9y vs EPTA dr2         hyper_DM      n_sigma=  0.012  method=kde
J1853+1303      NANOGrav 9y vs EPTA dr2         waveform_RN   n_sigma=  0.000  method=gaussian
J1853+1303      NANOGrav 9y vs EPTA dr2         timing        n_sigma=  0.000  method=gaussian


## Step 5 -- Tabulate with `summarize`

`summarize` returns a `pandas.DataFrame` sorted by pulsar / check / decreasing n_sigma. It is the same helper that produces the consistency-check table in the paper.

In [5]:
df = consistency.summarize(rows)
df[["pulsar", "pta_a", "pta_b", "check", "n_sigma", "p_value", "method", "n_params"]]

,pulsar,pta_a,pta_b,check,n_sigma,p_value,method,n_params
0,J1853+1303,NANOGrav 9y,EPTA dr2,hyper_DM,1.190677e-02,0.99050,kde,2
1,J1853+1303,NANOGrav 9y,EPTA dr2,hyper_RN,2.977061e-02,0.97625,kde,2
2,J1853+1303,NANOGrav 9y,EPTA dr2,timing,6.438055e-08,1.00000,gaussian,4
3,J1853+1303,NANOGrav 9y,EPTA dr2,waveform_RN,1.911741e-49,1.00000,gaussian,30


## What this looks like in production

The only difference between this notebook and a paper-grade consistency analysis is **where the samples come from**:

* For every per-PTA pulsar in `per_pta_psr`, build an FFTInt RN+DM model in `discovery` (or `enterprise_extensions`).
* Sample with `numpyro.infer.MCMC(NUTS(...))` (typically a few hundred warm-up + a few thousand samples; this is the dominant cost).
* Wrap each chain via `samples_to_mcsamples(samples, names=["log10_A", "gamma", ...])`.
* Feed the resulting `getdist.MCSamples` objects into `hyper_tension`, `waveform_tension`, `timing_tension` exactly as above.
* Optionally call `combined_vs_single_tension(combined_chain, single_chain, params=...)` to confirm the MetaPulsar restricted to one PTA agrees with that PTA's standalone analysis (this is the diagnostic shown in Figs. 8-9 of the paper).

When tensiometer is installed and the chains are long enough, `method="auto"` (the default) automatically uses tensiometer's KDE / normalising-flow estimators and only falls back to the closed-form Gaussian estimate if the upstream call fails. The reported `n_sigma` and `p_value` are normalised consistently across estimators, so the entries in the table above are directly comparable to a tensiometer-based table.